# Parcellated IS-RSA — Left-Wing Subjects: Affective Polarization

**Hypothesis 3**: Individual affective polarization modulates pairwise neural similarity.

**Behavioral score**: `affpol_thermo` — thermometer-based affective polarization index  
(in-party thermometer − out-party thermometer, range ~[−100, 100]).

**Method (IS-RSA)**: For each parcel and each condition grouping:
1. Average activation patterns across posts → one vector per (subject × parcel)
2. Build N×N **neural** similarity matrix (pairwise Pearson r)
3. Build N×N **behavioral** similarity matrix from pairwise `affpol_thermo` scores
4. Correlate matrices → IS-RSA r per parcel
5. Permutation test: shuffle subject labels on behavioral matrix (1,000 iterations)
6. FDR correction (Benjamini–Hochberg, q=0.05)

**Three analysis levels**:
- **Level A — All 4 conditions**: AntiLeft · AntiRight · ProLeft · ProRight
- **Level B — Agreed vs Disagreed (H1 level)**: Agreed={AntiRight+ProLeft} vs Disagreed={AntiLeft+ProRight}
- **Level C — Within Agreed (H2 level)**: AntiRight vs ProLeft

**Subjects**: Left-wing only (N=23) · **Atlas**: Schaefer 2018, 400 parcels, 7 networks (+Tian S3)

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import yabplot as yab
import yabplot.data as ydata

from yy_fmri_kit.event_isc.extraction import parcel
importlib.reload(parcel)
from yy_fmri_kit.event_isc.extraction.parcel import (
    Config,
    load_events,
    load_timeseries,
    extract_post_patterns,
    fdr_correct,
    results_to_dataframe,
    make_behavioral_rdm,
    compute_brain_behavior_rsa,
    permutation_test_brain_behavior,
)
from yy_fmri_kit.event_isc.contrast import merge_conditions, parcels_to_nifti

from yy_fmri_kit.visualization import pattern_analysis
importlib.reload(pattern_analysis)
from yy_fmri_kit.visualization.pattern_analysis import (
    plot_isc_parcels,
    plot_network_summary,
    plot_similarity_matrices,
    plot_brain_behavior_scatter,
    plot_brain_behavior_bar,
    plot_brain_map_interactive,
)

## 1. Configuration & Subject Selection

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────────
ROOT             = Path("/path/to/working/directory") # <-- UPDATE THIS PATH
BEHAVIORAL_CSV   = ROOT / "behavioral_analyses/data/250226/merged_behavioral_bids.csv"
POLARIZATION_CSV = ROOT / "behavioral_analyses/data/250226/political_polarization_scored.csv"
EVENTS_CSV       = ROOT / "behavioral_analyses/data/130426/summary_with_bids_ids.csv"
DATA_DIR         = ROOT / "data/derivatives/parcellated_tian"
ATLAS_NII        = ROOT / "data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz"
LABELS_TSV       = ROOT / "data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv"
OUTPUT_DIR       = ROOT / "data/derivatives/rsa/leftwing_isrsa/160426"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── left-wing subject selection (deduplicated) ───────────────────────────
behav_df = (
    pd.read_csv(BEHAVIORAL_CSV)
    .drop_duplicates(subset="bids_id", keep="first")
)
left_subs = behav_df.loc[behav_df["political_group"] == "left", "bids_id"].tolist()
print(f"Left-wing subjects (N={len(left_subs)}): {sorted(left_subs)}")

In [ ]:
cfg = Config(
    data_dir     = DATA_DIR,
    events_csv   = EVENTS_CSV,
    subjects     = left_subs,
    run_types    = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"],
    tr           = 1.0,
    shift_tr     = 4,
    tsv_glob     = "{subject}/{subject}_*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv",
    subject_col  = "bids_id",
    run_col      = "run",
    post_col     = "post_id",
    onset_col    = "onset_s",
    duration_col = "duration_s",
    n_perms      = 1000,
    fdr_q        = 0.05,
    seed         = 42,
)

## 2. Load Events, Timeseries & Extract Post Patterns

In [ ]:
events_df = load_events(cfg)
events_df = events_df.dropna(subset=["post_id"])
print(f"Events: {len(events_df)} rows")

ts_dict = load_timeseries(cfg)
parcel_names = ts_dict[next(iter(ts_dict))].columns.tolist()
print(f"Parcels: {len(parcel_names)}")

# patterns: {run_type: {post_id: array(n_subjects, n_parcels)}}
patterns = extract_post_patterns(ts_dict, events_df, cfg)
for rt, posts in patterns.items():
    print(f"  {rt}: {len(posts)} posts, {next(iter(posts.values())).shape[0]} subjects")

## 3. Build Behavioral Similarity Matrix (affpol_thermo)

In [ ]:
# Load polarization scores; normalise Code_tested to match subject_code in behav_df
pol_df = (
    pd.read_csv(POLARIZATION_CSV)
    .assign(subject_code=lambda d: d["Code_tested"].str.replace(
        r"YY_PL_0*(\d+)", r"YY_PL_\1", regex=True))
    .drop_duplicates(subset="subject_code", keep="first")
    [["subject_code", "affpol_thermo"]]
)

# Merge with bids_id via behav_df (already deduplicated)
pol_bids = pd.merge(
    pol_df,
    behav_df[["subject_code", "bids_id"]],
    on="subject_code",
    how="inner",
)

# affiliation dict: {bids_id: affpol_thermo}
affiliation = dict(zip(pol_bids["bids_id"], pol_bids["affpol_thermo"].astype(float)))
print(f"affpol_thermo loaded for {len(affiliation)} subjects")
print({s: affiliation[s] for s in sorted(affiliation)})

In [ ]:
# Ordered subject list as used by extract_post_patterns
subs_with_events = set(events_df[cfg.subject_col].unique())
subs_with_ts     = {s for s, _ in ts_dict.keys()}
valid_subs_order = [s for s in cfg.subjects if s in subs_with_ts and s in subs_with_events]

# Keep only subjects that also have polarization scores
bb_subjects = [s for s in valid_subs_order if s in affiliation]
bb_indices  = [valid_subs_order.index(s) for s in bb_subjects]
print(f"bb_subjects (N={len(bb_subjects)}): {bb_subjects}")

# Slice all pattern dicts to bb_subjects
def slice_patterns(pats, indices):
    return {rt: {pid: arr[indices, :] for pid, arr in posts.items()}
            for rt, posts in pats.items()}

patterns_bb = slice_patterns(patterns, bb_indices)

# N×N behavioral similarity matrix
beh_sim = make_behavioral_rdm(affiliation, bb_subjects)
print(f"Behavioral similarity matrix: {beh_sim.shape}")

## 4. Yabplot Brain-Map Helpers

Defined once; reused across all three analysis levels.

In [ ]:
_lh_surf, _rh_surf = ydata.get_surface_paths("midthickness", "bmesh")

ALL_VIEWS = [
    "left_lateral", "left_medial", "right_lateral", "right_medial",
    "superior", "inferior", "anterior", "posterior",
]

def brain_map(values, label, out_dir, *, cmap="coolwarm", vminmax=(None, None),
              nan_color=(0.92, 0.92, 0.92)):
    """Map parcel values → NIfTI → fsLR-32k surface → yabplot figure.
    Views: lateral + medial (both hemispheres), dorsal, ventral, frontal, occipital."""
    tmp = Path(tempfile.mktemp(suffix=".nii.gz"))
    parcels_to_nifti(values, parcel_names, ATLAS_NII, LABELS_TSV, tmp)
    lh_data, rh_data = yab.project_vol2surf(str(tmp), interpolation="nearest")
    tmp.unlink(missing_ok=True)
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    return yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views=ALL_VIEWS,
        cmap=cmap, vminmax=list(vminmax), nan_color=nan_color,
        figsize=(1600, 800), display_type="static",
        export_path=str(out_dir / f"{label}.png"),
    )

S3_TO_S1 = {
    "HIP-rh":  ["pHIP-rh"],
    "AMY-rh":  ["lAMY-rh",  "mAMY-rh"],
    "pTHA-rh": ["THA-DP-rh","THA-VP-rh"],
    "aTHA-rh": ["THA-VA-rh","THA-DA-rh"],
    "NAc-rh":  ["NAc-shell-rh","NAc-core-rh"],
    "GP-rh":   ["pGP-rh",  "aGP-rh"],
    "PUT-rh":  ["aPUT-rh", "pPUT-rh"],
    "CAU-rh":  ["aCAU-rh", "pCAU-rh"],
    "HIP-lh":  ["aHIP-lh", "pHIP-lh"],
    "AMY-lh":  ["lAMY-lh",  "mAMY-lh"],
    "pTHA-lh": ["THA-DP-lh","THA-VP-lh"],
    "aTHA-lh": ["THA-VA-lh","THA-DA-lh"],
    "NAc-lh":  ["NAc-shell-lh","NAc-core-lh"],
    "GP-lh":   ["pGP-lh",  "aGP-lh"],
    "PUT-lh":  ["aPUT-lh", "pPUT-lh"],
    "CAU-lh":  ["aCAU-lh", "pCAU-lh"],
}

def subcortical_map(values, label, out_dir, *, cmap="coolwarm", vminmax=(None, None),
                    nan_color=(0.92, 0.92, 0.92)):
    """Average Tian S3 sub-parcels into 16 S1 regions → yabplot plot_subcortical.
    Views: lateral + medial (both hemispheres), dorsal, ventral, frontal, occipital."""
    val_dict = {parcel_names[i]: float(values[i]) for i in range(len(values))}
    s1_dict  = {}
    for s1_name, s3_names in S3_TO_S1.items():
        sub_vals = [val_dict[n] for n in s3_names if n in val_dict]
        s1_dict[s1_name] = float(np.nanmean(sub_vals)) if sub_vals else float("nan")
    return yab.plot_subcortical(
        data=s1_dict, atlas="tian2020_s1",
        views=ALL_VIEWS,
        cmap=cmap, vminmax=list(vminmax), nan_color=nan_color,
        figsize=(1600, 800), display_type="static",
        export_path=str(out_dir / f"{label}_subcortical.png"),
    )

def run_brain_maps(results, out_dir, cmap="coolwarm"):
    """Full + FDR-sig cortical and subcortical maps for each condition."""
    all_r   = np.concatenate([results[rt]["r"].values for rt in results])
    cmax    = float(np.nanmax(np.abs(all_r)))
    vminmax = (-cmax, cmax)
    for rt, df in results.items():
        obs_r = df["r"].values
        sig_r = np.where(df["significant"].values.astype(bool), obs_r, np.nan)
        lbl   = rt.lower()
        print(f"\n=== {rt} — cortical (full) ===")
        brain_map(obs_r, f"{lbl}_full", out_dir, cmap=cmap, vminmax=vminmax)
        print(f"=== {rt} — subcortical (full) ===")
        subcortical_map(obs_r, f"{lbl}_full", out_dir, cmap=cmap, vminmax=vminmax)
        print(f"=== {rt} — cortical FDR-sig ({df['significant'].sum()} parcels) ===")
        brain_map(sig_r, f"{lbl}_sig", out_dir, cmap=cmap, vminmax=vminmax)
        print(f"=== {rt} — subcortical FDR-sig ===")
        subcortical_map(sig_r, f"{lbl}_sig", out_dir, cmap=cmap, vminmax=vminmax)

print("Yabplot helpers ready.")
print(f"Views: {ALL_VIEWS}")

## 5. IS-RSA & Visualisation Helpers

Convenience wrappers — keep each analysis level self-contained.

In [ ]:
def run_isrsa(pats_dict, beh_sim, subjects, cfg, parcel_names):
    """
    Run IS-RSA for every condition in pats_dict.

    Returns
    -------
    results : {condition: DataFrame(parcel, r, p_raw, p_fdr, significant)}
    nulls   : {condition: (obs_r, p_vals, null_dist)}
    """
    results, nulls = {}, {}
    for condition, pats in pats_dict.items():
        print(f"  {condition} ...", end=" ", flush=True)
        obs_r, p_vals, null = permutation_test_brain_behavior(
            pats, beh_sim, subjects, cfg)
        rejected, p_fdr = fdr_correct(p_vals, cfg.fdr_q)
        results[condition] = results_to_dataframe(
            parcel_names, obs_r, p_vals, rejected, p_fdr)
        nulls[condition]   = (obs_r, p_vals, null)
        print(f"{rejected.sum()} sig parcels | mean r={obs_r.mean():.3f}")
    return results, nulls


def save_isrsa(results, nulls, out_dir):
    """Save result CSVs and null-distribution .npy files."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for condition, df in results.items():
        df.to_csv(out_dir / f"{condition}.csv", index=False)
    for condition, (obs_r, p_vals, null_dist) in nulls.items():
        np.save(out_dir / f"{condition}_obs_r.npy",     obs_r)
        np.save(out_dir / f"{condition}_p_vals.npy",    p_vals)
        np.save(out_dir / f"{condition}_null_dist.npy", null_dist)
    print(f"Saved to {out_dir}")


def plot_summary(results, title, out_dir):
    """Bar + top-parcel + network-summary plots, saved as PNGs."""
    fig = plot_brain_behavior_bar(results, title=title)
    plt.tight_layout()
    plt.savefig(out_dir / "bar_conditions.png", dpi=150)
    plt.show()

    for condition, df in results.items():
        fig = plot_isc_parcels(df, run_type=condition, top_n=20, color="#9467bd")
        plt.tight_layout()
        plt.savefig(out_dir / f"{condition}_top_parcels.png", dpi=150)
        plt.show()

    fig = plot_network_summary(results, title=f"{title} — by network", ylabel="IS-RSA r")
    plt.tight_layout()
    plt.savefig(out_dir / "network_summary.png", dpi=150)
    plt.show()




def plot_sig_scatters(pats_dict, results, beh_sim, subjects, parcel_names, out_dir):
    """
    For each condition, plot neural vs behavioral similarity scatter
    for every FDR-significant parcel. Saves one PNG per parcel.
    """
    for condition, df in results.items():
        sig_df = df[df["significant"] == 1]
        if sig_df.empty:
            print(f"{condition}: no significant parcels — skipping scatter")
            continue
        # compute neural similarity matrix (N×N×P) for this condition
        _, neural_sim = compute_brain_behavior_rsa(
            pats_dict[condition], beh_sim, subjects)
        for _, row in sig_df.iterrows():
            p_idx = parcel_names.index(row["parcel"])
            safe = row["parcel"].replace("/", "_")
            fig = plot_similarity_matrices(
                neural_sim, beh_sim, subjects,
                p_idx, parcel_name=row["parcel"], run_type=condition)
            plt.tight_layout()
            plt.savefig(out_dir / f"{condition}_{safe}_simmat.png", dpi=150)
            plt.show()
            fig = plot_brain_behavior_scatter(
                neural_sim, beh_sim, subjects,
                p_idx, parcel_name=row["parcel"], run_type=condition)
            plt.tight_layout()
            plt.savefig(out_dir / f"{condition}_{safe}_scatter.png", dpi=150)
            plt.show()


print("Helpers ready.")

---
## Level A — All 4 Conditions

IS-RSA run separately for AntiLeft, AntiRight, ProLeft, ProRight.

In [ ]:
OUT_A = OUTPUT_DIR / "all_conditions"
OUT_A.mkdir(exist_ok=True)

print("Running Level A — all 4 conditions:")
results_A, nulls_A = run_isrsa(patterns_bb, beh_sim, bb_subjects, cfg, parcel_names)
save_isrsa(results_A, nulls_A, OUT_A)

In [ ]:
pd.DataFrame({
    rt: {"n_sig": df["significant"].sum(),
         "mean_r": df["r"].mean().round(3),
         "max_r":  df["r"].max().round(3)}
    for rt, df in results_A.items()
}).T

In [ ]:
plot_summary(results_A,
    title="IS-RSA: affpol_thermo — all 4 conditions",
    out_dir=OUT_A)

In [ ]:
# Scatter: neural vs behavioral similarity for each FDR-significant parcel — Level A
plot_sig_scatters(patterns_bb, results_A, beh_sim, bb_subjects, parcel_names, OUT_A)

In [ ]:
run_brain_maps(results_A, OUT_A)

In [ ]:
plot_brain_map_interactive(
    results_A, output_dir=str(OUT_A),
    p_thresholds=[None, 0.05], p_col="p_raw", n_rois=400,
)

---
## Level B — Agreed vs Disagreed  *(H1 level)*

**Agreed** = AntiRight + ProLeft (content left-wing subjects agree with)  
**Disagreed** = AntiLeft + ProRight (content they disagree with)  
IS-RSA run separately for each group.

In [ ]:
CONDITION_MAP_B = {
    "agreed":    ["AntiRight", "ProLeft"],
    "disagreed": ["AntiLeft",  "ProRight"],
}

merged_B = merge_conditions(patterns_bb, CONDITION_MAP_B)
print("Post counts after merging:")
for cond, posts in merged_B.items():
    print(f"  {cond}: {len(posts)} posts")

In [ ]:
OUT_B = OUTPUT_DIR / "agreed_vs_disagreed"
OUT_B.mkdir(exist_ok=True)

print("Running Level B — agreed vs disagreed:")
results_B, nulls_B = run_isrsa(merged_B, beh_sim, bb_subjects, cfg, parcel_names)
save_isrsa(results_B, nulls_B, OUT_B)

In [ ]:
pd.DataFrame({
    cond: {"n_sig": df["significant"].sum(),
           "mean_r": df["r"].mean().round(3),
           "max_r":  df["r"].max().round(3)}
    for cond, df in results_B.items()
}).T

In [ ]:
plot_summary(results_B,
    title="IS-RSA: affpol_thermo — agreed vs disagreed",
    out_dir=OUT_B)

In [ ]:
# Scatter: neural vs behavioral similarity for each FDR-significant parcel — Level B
plot_sig_scatters(merged_B, results_B, beh_sim, bb_subjects, parcel_names, OUT_B)

In [ ]:
run_brain_maps(results_B, OUT_B)

In [ ]:
plot_brain_map_interactive(
    results_B, output_dir=str(OUT_B),
    p_thresholds=[None, 0.05], p_col="p_raw", n_rois=400,
)

---
## Level C — Within Agreed: AntiRight vs ProLeft  *(H2 level)*

Both conditions are within the agreed category:  
- **AntiRight** — outgroup-derogating content  
- **ProLeft** — ingroup-favoring content  
IS-RSA run separately for each.

In [ ]:
within_agreed = {
    "AntiRight": patterns_bb["AntiRight"],
    "ProLeft":   patterns_bb["ProLeft"],
}

OUT_C = OUTPUT_DIR / "within_agreed"
OUT_C.mkdir(exist_ok=True)

print("Running Level C — within agreed (AntiRight vs ProLeft):")
results_C, nulls_C = run_isrsa(within_agreed, beh_sim, bb_subjects, cfg, parcel_names)
save_isrsa(results_C, nulls_C, OUT_C)

In [ ]:
pd.DataFrame({
    cond: {"n_sig": df["significant"].sum(),
           "mean_r": df["r"].mean().round(3),
           "max_r":  df["r"].max().round(3)}
    for cond, df in results_C.items()
}).T

In [ ]:
plot_summary(results_C,
    title="IS-RSA: affpol_thermo — within agreed",
    out_dir=OUT_C)

In [ ]:
# Scatter: neural vs behavioral similarity for each FDR-significant parcel — Level C
plot_sig_scatters(within_agreed, results_C, beh_sim, bb_subjects, parcel_names, OUT_C)

In [ ]:
run_brain_maps(results_C, OUT_C)

In [ ]:
plot_brain_map_interactive(
    results_C, output_dir=str(OUT_C),
    p_thresholds=[None, 0.05], p_col="p_raw", n_rois=400,
)